# Segment C — Multi-Turn State + Script Transition
### Module 1: Claude API & Ecosystem Foundations

**Goal:** turn Segment B's one-shot `run_agent()` function into something that
remembers prior turns, then move that code out of the notebook and into a proper
`.py` module — because Segment D's capstone needs to run as a standalone script,
not a notebook cell.

We're doing this transition deliberately and visibly, rather than just handing you
a finished script, because the *reason* for the transition matters: notebooks are
great for exploring, but a CLI agent that someone runs from a terminal needs to be
a script.

## 1. Setup — same as Segment B

In [ ]:
import os
import json
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
from anthropic import Anthropic

os.environ["ANTHROPIC_API_KEY"] = "sk-..."

client = Anthropic()
MODEL = "claude-sonnet-4-6"


def search_wikipedia(query: str, limit: int = 3) -> str:
    url = "https://en.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
        "action": "query", "list": "search", "srsearch": query,
        "format": "json", "srlimit": limit,
    })
    req = urllib.request.Request(url, headers={"User-Agent": "Module1-ResearchCompanion/1.0"})
    with urllib.request.urlopen(req, timeout=10) as resp:
        data = json.loads(resp.read())
    results = data.get("query", {}).get("search", [])
    if not results:
        return f"No Wikipedia results found for '{query}'."
    lines = []
    for r in results:
        snippet = r["snippet"].replace('<span class="searchmatch">', "").replace("</span>", "")
        lines.append(f"- {r['title']}: {snippet}")
    return "\n".join(lines)


def search_arxiv(query: str, max_results: int = 3) -> str:
    url = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode({
        "search_query": f"all:{query}", "start": 0, "max_results": max_results,
    })
    with urllib.request.urlopen(url, timeout=10) as resp:
        raw = resp.read()
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    root = ET.fromstring(raw)
    entries = root.findall("atom:entry", ns)
    if not entries:
        return f"No arXiv results found for '{query}'."
    lines = []
    for e in entries:
        title = e.find("atom:title", ns).text.strip().replace("\n", " ")
        authors = [a.find("atom:name", ns).text for a in e.findall("atom:author", ns)]
        summary = e.find("atom:summary", ns).text.strip().replace("\n", " ")[:200]
        lines.append(f"- {title} ({', '.join(authors)}): {summary}...")
    return "\n".join(lines)


TOOLS = [
    {
        "name": "search_wikipedia",
        "description": "Search Wikipedia for general background on a topic, person, place, or concept.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "The search query."}},
            "required": ["query"],
        },
    },
    {
        "name": "search_arxiv",
        "description": "Search arXiv for academic papers on technical or research topics.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "The search query."}},
            "required": ["query"],
        },
    },
]

TOOL_FUNCTIONS = {"search_wikipedia": search_wikipedia, "search_arxiv": search_arxiv}

print("Setup complete.")

Setup complete.


## 2. The problem with `run_agent()` from Segment B

Segment B's function took a single string and returned a single string. It built
a fresh `messages` list every call — meaning each call had **zero memory** of any
previous call. Try it:

In [2]:
def run_agent_v1(user_message: str, max_iterations: int = 5) -> str:
    messages = [{"role": "user", "content": user_message}]
    for _ in range(max_iterations):
        response = client.messages.create(
            model=MODEL, max_tokens=800, tools=TOOLS, messages=messages,
        )
        if response.stop_reason != "tool_use":
            return response.content[0].text
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            output = TOOL_FUNCTIONS[block.name](**block.input)
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})
        messages.append({"role": "user", "content": tool_results})
    return "Reached max_iterations."


print(run_agent_v1("My name is Priya. Remember that."))
print()
print(run_agent_v1("What's my name?"))

Got it, **Priya**! I'll remember your name for the duration of our conversation. How can I help you today?

I don't have access to any personal information about you, so I don't know your name. You haven't shared it with me in our conversation. Feel free to introduce yourself if you'd like! 😊


Claude has no idea — because each call started a brand new `messages` list. This
isn't a bug, it's the direct consequence of everything from this morning's lecture:
the array IS the memory, and we just threw the array away between calls.

**Fix:** keep the `messages` list alive across calls, owned by something that
persists — a class instance is the natural fit.

## 3. Refactor into a stateful `ResearchCompanion` class

In [3]:
class ResearchCompanion:
    """
    A single-agent research assistant that keeps conversation history across
    multiple turns. Tools are hardcoded for now — Module 2 replaces this with
    tools served over MCP.
    """

    def __init__(self, model: str = MODEL, max_iterations: int = 5):
        self.model = model
        self.max_iterations = max_iterations
        self.messages = []  # this list persists across .ask() calls

    def ask(self, user_message: str) -> str:
        self.messages.append({"role": "user", "content": user_message})

        for _ in range(self.max_iterations):
            response = client.messages.create(
                model=self.model,
                max_tokens=800,
                tools=TOOLS,
                messages=self.messages,
            )

            if response.stop_reason != "tool_use":
                final_text = response.content[0].text
                self.messages.append({"role": "assistant", "content": response.content})
                return final_text

            self.messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                print(f"  [tool call] {block.name}({block.input})")
                output = TOOL_FUNCTIONS[block.name](**block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                })
            self.messages.append({"role": "user", "content": tool_results})

        return "Reached max_iterations without a final answer."

## 4. Try it — now with real memory

In [4]:
companion = ResearchCompanion()

print(companion.ask("My name is Priya. Remember that."))
print()
print(companion.ask("What's my name?"))

I'll make sure to remember that your name is Priya! However, I should be transparent with you — I don't have the ability to store or retain information between conversations. So while I know your name is **Priya** for the duration of this current conversation, if we start a new chat, I won't have that information.

Is there anything I can help you with today, Priya?

Your name is **Priya**! You told me at the start of our conversation. 😊

Is there anything else I can help you with?


In [5]:
# A more realistic multi-turn research flow:
print(companion.ask("Give me a quick overview of multi-agent systems."))
print()
print(companion.ask("Is there recent academic research specifically on that?"))

  [tool call] search_wikipedia({'query': 'multi-agent systems'})
  [tool call] search_arxiv({'query': 'multi-agent systems overview'})
Here's a quick overview of **Multi-Agent Systems (MAS)**, Priya:

---

### 🤖 What is a Multi-Agent System?
A **Multi-Agent System (MAS)** is a computational system made up of multiple interacting intelligent **agents** — entities that can perceive their environment, make decisions, and take actions to achieve goals.

---

### 🔑 Key Characteristics
- **Autonomy** – Each agent operates independently, making its own decisions.
- **Interaction** – Agents communicate and coordinate with one another.
- **Decentralization** – There's no single point of control; intelligence is distributed.
- **Emergence** – Complex behaviors can arise from simple agent interactions.

---

### 🌐 Core Concepts
- **Cooperation** – Agents work together to achieve shared goals.
- **Competition** – Agents may also compete, drawing from **game theory**.
- **Communication** – A crucia

Notice the second question — "Is there recent academic research specifically on
**that**?" — only makes sense because `self.messages` carried the topic forward.
This is the same mechanism as Segment A's manual multi-turn example, just owned
by the class instead of by us typing it out by hand each time.

## 5. A quick look at what's accumulating

It's worth peeking at `companion.messages` directly once, so the growth of this
list over a conversation is visible, not just theoretical.

In [6]:
print(f"Total messages in history: {len(companion.messages)}")
for i, m in enumerate(companion.messages):
    role = m["role"]
    content = m["content"]
    if isinstance(content, str):
        preview = content[:60]
    else:
        preview = f"[{len(content)} content block(s)]"
    print(f"{i}: {role} — {preview}")

Total messages in history: 12
0: user — My name is Priya. Remember that.
1: assistant — [1 content block(s)]
2: user — What's my name?
3: assistant — [1 content block(s)]
4: user — Give me a quick overview of multi-agent systems.
5: assistant — [3 content block(s)]
6: user — [2 content block(s)]
7: assistant — [1 content block(s)]
8: user — Is there recent academic research specifically on that?
9: assistant — [3 content block(s)]
10: user — [2 content block(s)]
11: assistant — [1 content block(s)]


## 6. Moving to a script

Everything above works fine in a notebook, but Segment D's capstone is a CLI tool
someone runs with `python research_companion.py` — that needs to live in a `.py`
file, not notebook cells.

**What we're moving, and why:**

| Piece | Stays in notebook? | Goes to script? |
|---|---|---|
| `search_wikipedia`, `search_arxiv` | No | Yes — these are the tool implementations |
| `TOOLS`, `TOOL_FUNCTIONS` | No | Yes — tool registry |
| `ResearchCompanion` class | No | Yes — the agent itself |
| A `main()` that runs a CLI loop | N/A | Yes — new, doesn't exist yet |

The cell below writes everything out to `research_companion.py` using a Jupyter
magic command, so you can see the exact same code land in a real file. In your own
workflow you'd normally just write the `.py` file directly in VS Code — we're using
the `%%writefile` trick here only so the notebook and the script stay visibly in
sync for teaching purposes.

In [7]:
%%writefile research_companion.py
"""
Research Companion v1 — single agent, hardcoded tools, no framework.
Module 1 capstone. Run with: python research_companion.py
"""
import json
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
from anthropic import Anthropic

MODEL = "claude-sonnet-4-6"
client = Anthropic()


def search_wikipedia(query: str, limit: int = 3) -> str:
    """Search Wikipedia for general background on a topic."""
    url = "https://en.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
        "action": "query", "list": "search", "srsearch": query,
        "format": "json", "srlimit": limit,
    })
    req = urllib.request.Request(url, headers={"User-Agent": "Module1-ResearchCompanion/1.0"})
    with urllib.request.urlopen(req, timeout=10) as resp:
        data = json.loads(resp.read())
    results = data.get("query", {}).get("search", [])
    if not results:
        return f"No Wikipedia results found for '{query}'."
    lines = []
    for r in results:
        snippet = r["snippet"].replace('<span class="searchmatch">', "").replace("</span>", "")
        lines.append(f"- {r['title']}: {snippet}")
    return "\n".join(lines)


def search_arxiv(query: str, max_results: int = 3) -> str:
    """Search arXiv for academic papers."""
    url = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode({
        "search_query": f"all:{query}", "start": 0, "max_results": max_results,
    })
    with urllib.request.urlopen(url, timeout=10) as resp:
        raw = resp.read()
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    root = ET.fromstring(raw)
    entries = root.findall("atom:entry", ns)
    if not entries:
        return f"No arXiv results found for '{query}'."
    lines = []
    for e in entries:
        title = e.find("atom:title", ns).text.strip().replace("\n", " ")
        authors = [a.find("atom:name", ns).text for a in e.findall("atom:author", ns)]
        summary = e.find("atom:summary", ns).text.strip().replace("\n", " ")[:200]
        lines.append(f"- {title} ({', '.join(authors)}): {summary}...")
    return "\n".join(lines)


TOOLS = [
    {
        "name": "search_wikipedia",
        "description": "Search Wikipedia for general background on a topic, person, place, or concept. Good for established facts. Not useful for very recent events or cutting-edge research.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "The search query."}},
            "required": ["query"],
        },
    },
    {
        "name": "search_arxiv",
        "description": "Search arXiv for academic papers. Good for cutting-edge research and technical methods. Not useful for general background.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "The search query."}},
            "required": ["query"],
        },
    },
]

TOOL_FUNCTIONS = {"search_wikipedia": search_wikipedia, "search_arxiv": search_arxiv}


class ResearchCompanion:
    """A single-agent research assistant with hardcoded tools and persistent history."""

    def __init__(self, model: str = MODEL, max_iterations: int = 5):
        self.model = model
        self.max_iterations = max_iterations
        self.messages = []

    def ask(self, user_message: str) -> str:
        self.messages.append({"role": "user", "content": user_message})

        for _ in range(self.max_iterations):
            response = client.messages.create(
                model=self.model, max_tokens=800, tools=TOOLS, messages=self.messages,
            )

            if response.stop_reason != "tool_use":
                self.messages.append({"role": "assistant", "content": response.content})
                return response.content[0].text

            self.messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                print(f"  [tool call] {block.name}({block.input})")
                output = TOOL_FUNCTIONS[block.name](**block.input)
                tool_results.append({
                    "type": "tool_result", "tool_use_id": block.id, "content": output,
                })
            self.messages.append({"role": "user", "content": tool_results})

        return "Reached max_iterations without a final answer."


def main():
    print("Research Companion v1 — type 'quit' to exit.")
    companion = ResearchCompanion()
    while True:
        user_input = input("\nYou: ").strip()
        if user_input.lower() in {"quit", "exit"}:
            break
        if not user_input:
            continue
        answer = companion.ask(user_input)
        print(f"\nCompanion: {answer}")


if __name__ == "__main__":
    main()

Overwriting research_companion.py


## 7. Sanity-check the script runs

We can run it from inside the notebook with a shell command, just to confirm it
executes — though in the actual class, the natural move is to switch your screen
to a terminal in VS Code at this point and run it from there directly.

In [8]:
# A non-interactive smoke test: pipe a couple of inputs in and confirm no crash.
import subprocess

test_input = "What is agentic AI?\nquit\n"
result = subprocess.run(
    ["python", "research_companion.py"],
    input=test_input, capture_output=True, text=True, timeout=60,
)
print(result.stdout[-1500:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

ep.
- **Agentic AI**: Given a high-level goal, the system *plans*, *acts*, *observes the results*, and *iterates* — often looping through many steps autonomously before delivering a final outcome.

---

### 🛠️ Real-World Examples

- **Autonomous research agents** that search the web, read papers, and summarize findings
- **Coding agents** that write, test, debug, and deploy code
- **Healthcare agents** that query databases, analyze records, and support clinical decisions
- **Enterprise agents** that manage workflows, retrieve documents, and interact with business systems

---

### ⚠️ Key Challenges & Risks

As highlighted in recent research, agentic AI introduces unique concerns:
- **Safety & Security**: Systems with shell execution, file system access, and network access can cause real-world harm if misused or compromised
- **Trustworthiness**: Ensuring agents behave reliably across long, multi-step tasks
- **Privacy**: Agents handling sensitive data (e.g., in healthcare or enterprise

## Recap

- A one-shot function has no memory between calls — the `messages` list dies with the function call
- Wrapping the loop in a class with `self.messages` is enough to fix that — nothing exotic, just an object that persists
- We moved tool functions, tool registry, and the agent class into `research_companion.py` because Segment D's capstone needs to run as a CLI script, not notebook cells
- `%%writefile` was a teaching convenience here — in your own work you'd just write the `.py` file directly

Next: Segment D — finish the capstone, optionally add a third tool, and wrap up Module 1.